# Aetheria 2099 — 소형 모델 파인튜닝 (Colab 무료 T4)

Qwen2.5-3B-Instruct 를 **QLoRA(4bit)** 로 파인튜닝해, 증류한 `train.jsonl` 을 학습시킨다. 학습 후 **GGUF** 로 내보내 로컬 Ollama에서 구동.

**먼저:** 상단 메뉴 → 런타임 → 런타임 유형 변경 → **T4 GPU** 선택. 그다음 셀을 위에서부터 순서대로 실행.

## 셀 1 — Unsloth 설치 (T4에서 QLoRA를 빠르게)

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes

## 셀 2 — 데이터셋 업로드
로컬 `ml/train.jsonl`(학습 285개)을 업로드. 평가용 `eval.jsonl`(홀드아웃 40개)은 **넣지 않는다** — 학습 후 로컬에서 `eval.mjs`로 정직한 baseline 측정.

In [ ]:
from google.colab import files
up = files.upload()  # train.jsonl 선택
DATA_PATH = "train.jsonl"

## 셀 3 — 베이스 모델 로드 (Qwen2.5-**7B**, 4bit)

3B→7B 상향: 한국어·일관성이 뚜렷이 개선된다. 무료 **T4(16GB)** 에서 QLoRA 4bit로 학습 가능하나 3B보다 2~3배 느리다. (VRAM 부족 뜨면 MAX_SEQ를 3072로 낮춘다.)

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ = 3072  # 예제는 앵커 포함 ~2k토큰이라 4096이면 충분
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
    dtype=None,
)

## 셀 4 — LoRA 어댑터 부착 (작은 어댑터만 학습)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

## 셀 5 — 데이터 로드 + 채팅 템플릿 (Qwen = ChatML)

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")
ds = load_dataset("json", data_files=DATA_PATH, split="train")

def fmt(ex):
    text = tokenizer.apply_chat_template(ex["messages"], tokenize=False, add_generation_prompt=False)
    return {"text": text}

ds = ds.map(fmt)
print("예제 수:", len(ds))
print(ds[0]["text"][:600])

## 셀 6 — 학습 (SFT, assistant JSON만 손실 계산)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

# 최신 TRL은 dataset_text_field / max_seq_length 를 SFTConfig 안에 둔다.
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=10,
        num_train_epochs=2,
        learning_rate=2e-4,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
trainer.train()

## 셀 7 — 빠른 확인 (학습된 모델이 JSON을 뱉는지)

In [ ]:
FastLanguageModel.for_inference(model)
sample = ds[0]["messages"][:2]  # system+user만
prompt = tokenizer.apply_chat_template(sample, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
out = model.generate(**inputs, max_new_tokens=512, temperature=0.9, do_sample=True)
print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

## 셀 8 — GGUF 내보내기 + Google Drive 자동 저장

세션이 끊겨도 안전하게 **Drive `MyDrive/aetheria-model/` 에 자동 복사**한다. (첫 실행 시 Drive 접근 권한 승인창이 뜬다.)

In [ ]:
# GGUF로 내보내고 Google Drive에 자동 저장
from google.colab import drive
drive.mount("/content/drive")

model.save_pretrained_gguf("aetheria-qwen3b", tokenizer, quantization_method="q4_k_m")

import shutil, os, glob
# Unsloth는 *_gguf 폴더에 넣기도 하므로 재귀로 찾는다.
cands = glob.glob("**/*.gguf", recursive=True)
print("찾은 gguf:", cands)
src = next((p for p in cands if "Q4_K_M" in p or "q4_k_m" in p), cands[0])
SAVE_DIR = "/content/drive/MyDrive/aetheria-model"
os.makedirs(SAVE_DIR, exist_ok=True)
dst = os.path.join(SAVE_DIR, os.path.basename(src))
shutil.copy(src, dst)
print("✅ Drive에 저장 완료:", dst)
print("   크기: %.1f MB" % (os.path.getsize(dst)/1e6))

# (선택) 브라우저로도 바로 받고 싶으면 주석 해제:
# from google.colab import files; files.download(src)